{"username":"bebseee","key":"5325e188491c79d44b6a534b2d96b3f2"}

In [1]:
# ============================================
# [Step 1] 환경 세팅 및 데이터 다운로드 (Modern Ver.)
# ============================================
import os
import getpass

# 1. Kaggle API 인증 정보 입력
# 참가자가 본인의 Kaggle Username과 Key를 직접 입력하게 합니다. (보안성↑)
KAG_USER = getpass.getpass('Kaggle Username을 입력하세요: ')
KAG_KEY = getpass.getpass('Kaggle API Key를 입력하세요: ')

os.environ['KAGGLE_USERNAME'] = KAG_USER
os.environ['KAGGLE_KEY'] = KAG_KEY

# 2. 필수 라이브러리 설치
!pip install pillow-heif -q
!pip install kaggle -q

# 3. 데이터셋 다운로드 및 압축 해제
print("📥 데이터셋 다운로드 중...")
# 데이터셋 API 주소 사용
!kaggle competitions download -c aikuthon11th

print("📦 압축 해제 중...")
# 기존 파일이 있다면 지우고 깔끔하게 해제
if os.path.exists('/content/dataset'):
    import shutil
    shutil.rmtree('/content/dataset')

!unzip -q aikuthon11th.zip -d /content/dataset

print("✅ 모든 준비가 완료되었습니다!")

Kaggle Username을 입력하세요: ··········
Kaggle API Key를 입력하세요: ··········
📥 데이터셋 다운로드 중...
aikuthon11th.zip: Skipping, found more recently modified local copy (use --force to force download)
📦 압축 해제 중...
✅ 모든 준비가 완료되었습니다!


In [2]:
# ============================================
# [Step 2] 라이브러리 임포트 및 Dataset 정의 (성능 최적화 버전)
# ============================================
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import pillow_heif

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# HEIC 파일 지원 설정
pillow_heif.register_heif_opener()

# 1. 재현성을 위한 시드 고정
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")

# 2. --- Dataset 정의 (Albumentations 호환을 위해 Numpy 변환 추가) ---
class PhoneDataset(Dataset):
    def __init__(self, file_list, labels, transform=None):
        self.file_list = file_list
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        # Albumentations는 Numpy 배열을 입력으로 받습니다
        img = np.array(Image.open(self.file_list[idx]).convert('RGB'))
        if self.transform:
            # Albumentations 호출 방식 적용
            img = self.transform(image=img)['image']
        return img, self.labels[idx]

class TestDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.file_list[idx]).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img

# 3. --- Train/Val 데이터 수집 (Galaxy=0, iPhone=1) ---
train_files_all, train_labels_all = [], []
train_dir = '/content/dataset/train'

for f in sorted(os.listdir(os.path.join(train_dir, 'Galaxy'))):
    train_files_all.append(os.path.join(train_dir, 'Galaxy', f))
    train_labels_all.append(0)

for f in sorted(os.listdir(os.path.join(train_dir, 'iPhone'))):
    train_files_all.append(os.path.join(train_dir, 'iPhone', f))
    train_labels_all.append(1)

# 8:2 비율로 클래스 균형을 유지하며 분할 (stratify 필수)
train_files, val_files, train_labels, val_labels = train_test_split(
    train_files_all, train_labels_all, test_size=0.2, random_state=42, stratify=train_labels_all
)

# --- Test 데이터 수집 ---
test_dir = '/content/dataset/test'
test_fnames = sorted(os.listdir(test_dir))
test_files = [os.path.join(test_dir, f) for f in test_fnames]

# 4. --- 전처리 및 증강 파이프라인 (최종 전략 반영) ---
train_transform = A.Compose([
    A.OneOf([
        # 사물의 크기 변화 대응 (Scale-invariance)
        A.RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0), p=1.0),
        # 원본 질감 보존 (Patch-based / Unseen Device 대응)
        A.RandomCrop(height=224, width=224, p=1.0),
    ], p=1.0),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),

    # 디지털 변형 시뮬레이션 (압축 노이즈, 선예도)
    A.OneOf([
        A.ImageCompression(quality_lower=60, quality_upper=100, p=1.0),
        A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
        A.Sharpen(alpha=(0.2, 0.5), p=1.0),
    ], p=0.4),

    # 색감 보존 (밝기/대비 위주, Hue는 건드리지 않음)
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),

    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(height=224, width=224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# 5. --- WeightedRandomSampler 설정 (아이폰 데이터 부족 해결) ---
class_counts = np.bincount(train_labels)
class_weights = 1. / class_counts
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True # 소수 클래스 중복 추출 허용
)

# 6. --- DataLoader 연결 (sampler 사용 시 shuffle=True 제거) ---
g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(
    PhoneDataset(train_files, train_labels, train_transform),
    batch_size=32,
    sampler=sampler, # 샘플러 적용
    generator=g
)
val_loader = DataLoader(
    PhoneDataset(val_files, val_labels, val_transform),
    batch_size=32,
    shuffle=False
)
test_loader = DataLoader(
    TestDataset(test_files, val_transform),
    batch_size=32,
    shuffle=False
)

print(f"📊 Train: {len(train_files)}장 | Val: {len(val_files)}장 | Test: {len(test_files)}장")
print(f"⚖️ 클래스별 가중치: {class_weights} (0: Galaxy, 1: iPhone)")

🖥️ Device: cuda
📊 Train: 694장 | Val: 174장 | Test: 587장
⚖️ 클래스별 가중치: [0.00653595 0.00184843] (0: Galaxy, 1: iPhone)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-70277/716374408.py:106: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=60, quality_upper=100, p=1.0),
/tmp/ipython-input-70277/716374408.py:107: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),


In [3]:
# ============================================
# [Step 3] 모델 정의 (EfficientNet-B0 + Label Smoothing)
# ============================================

from transformers import ViTForImageClassification, ViTImageProcessor
import torch.nn as nn # Ensure nn is imported if not already

set_seed(42)

# 1. 모델 선언:
# model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

# 2. 마지막 분류 레이어 교체 (출력 1개: Galaxy vs iPhone)
# EfficientNet은 classifier[1]이 최종 선형 레이어입니다.
# ViT 모델은 model.classifier가 직접 최종 선형 레이어입니다.
model.classifier = nn.Linear(model.classifier.in_features, 1)
model = model.to(device)

# 3. 손실 함수: Label Smoothing 적용
# Unseen Device 대응을 위해 정답을 0.9, 오답을 0.1로 부드럽게 인식하게 합니다.
# PyTorch 1.10+ 버전은 label_smoothing 인자를 직접 지원합니다.
criterion = nn.BCEWithLogitsLoss(pos_weight=None)
LABEL_SMOOTHING = 0.1

# 4. 옵티마이저 & 스케줄러
# AdamW는 일반화 성능이 더 뛰어나며, CosineAnnealing은 부드러운 수렴을 돕습니다.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print("🚀 VIT 모델 및 Label Smoothing 세팅 완료!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

🚀 VIT 모델 및 Label Smoothing 세팅 완료!


In [7]:
# ============================================
# [Step 4] 학습 및 검증 (Label Smoothing & Scheduler 반영)
# ============================================
EPOCHS = 30
LABEL_SMOOTHING = 0.1 # 우리가 정한 0.1 수치 적용

for epoch in range(EPOCHS):
    # --- Train 단계 ---
    model.train()
    train_loss = 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.float().to(device)

        # [전략] Label Smoothing 수동 적용
        # 1(iPhone) -> 0.95, 0(Galaxy) -> 0.05로 변환하여 과적합 방지
        smoothed_lbls = lbls * (1 - LABEL_SMOOTHING) + 0.5 * LABEL_SMOOTHING

        optimizer.zero_grad()
        outputs = model(imgs).logits.squeeze()

        # 부드러워진 라벨로 Loss 계산
        loss = criterion(outputs, smoothed_lbls)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # 스케줄러 업데이트 (Step 3에서 정의한 CosineAnnealing 적용)
    scheduler.step()

    # --- Validation 단계 ---
    model.eval()
    val_preds, val_trues = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device)
            # 예측 확률값 추출 (0~1 사이)
            probs = torch.sigmoid(model(imgs).logits.squeeze())

            val_preds.extend(probs.cpu().numpy())
            val_trues.extend(lbls.numpy())

    # 대회 메트릭인 AUC-ROC 계산
    val_auc = roc_auc_score(val_trues, val_preds)

    # 현재 학습률 확인 (디버깅용)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{EPOCHS}] | LR: {current_lr:.6f} | Train Loss: {train_loss/len(train_loader):.4f} | Val AUC: {val_auc:.4f}")

Epoch [1/30] | LR: 0.000098 | Train Loss: 0.3980 | Val AUC: 0.8996
Epoch [2/30] | LR: 0.000090 | Train Loss: 0.3544 | Val AUC: 0.9170
Epoch [3/30] | LR: 0.000079 | Train Loss: 0.3385 | Val AUC: 0.9226
Epoch [4/30] | LR: 0.000065 | Train Loss: 0.3158 | Val AUC: 0.9205
Epoch [5/30] | LR: 0.000050 | Train Loss: 0.3099 | Val AUC: 0.9207
Epoch [6/30] | LR: 0.000035 | Train Loss: 0.2995 | Val AUC: 0.9247
Epoch [7/30] | LR: 0.000021 | Train Loss: 0.2928 | Val AUC: 0.9255
Epoch [8/30] | LR: 0.000010 | Train Loss: 0.2958 | Val AUC: 0.9284
Epoch [9/30] | LR: 0.000002 | Train Loss: 0.2960 | Val AUC: 0.9301
Epoch [10/30] | LR: 0.000000 | Train Loss: 0.3020 | Val AUC: 0.9286
Epoch [11/30] | LR: 0.000002 | Train Loss: 0.3075 | Val AUC: 0.9286
Epoch [12/30] | LR: 0.000010 | Train Loss: 0.3000 | Val AUC: 0.9286
Epoch [13/30] | LR: 0.000021 | Train Loss: 0.2951 | Val AUC: 0.9303
Epoch [14/30] | LR: 0.000035 | Train Loss: 0.2900 | Val AUC: 0.9269
Epoch [15/30] | LR: 0.000050 | Train Loss: 0.2873 | Val A

In [8]:
# ============================================
# [Step 4] 학습 및 검증 (Validation Loss 추가 버전)
# ============================================
EPOCHS = 30
LABEL_SMOOTHING = 0.1

best_val_auc = 0.0

for epoch in range(EPOCHS):
    # --- Train 단계 ---
    model.train()
    train_loss = 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.float().to(device)

        # Label Smoothing 적용
        smoothed_lbls = lbls * (1 - LABEL_SMOOTHING) + 0.5 * LABEL_SMOOTHING

        optimizer.zero_grad()
        outputs = model(imgs).logits.squeeze()

        loss = criterion(outputs, smoothed_lbls)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # 스케줄러 업데이트
    scheduler.step()

    # --- Validation 단계 ---
    model.eval()
    val_loss = 0  # Validation Loss 초기화
    val_preds, val_trues = [], []

    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.float().to(device) # Loss 계산을 위해 lbls도 device로 이동

            outputs = model(imgs).logits.squeeze()

            # Validation Loss 계산 (평가 시에는 Smoothing 미적용 원본 라벨 사용이 일반적입니다)
            v_loss = criterion(outputs, lbls)
            val_loss += v_loss.item()

            # AUC 계산을 위한 확률값 추출
            probs = torch.sigmoid(outputs)
            val_preds.extend(probs.cpu().numpy())
            val_trues.extend(lbls.cpu().numpy())

    # 메트릭 계산
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_auc = roc_auc_score(val_trues, val_preds)

    # 최고 성능 모델 저장
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), 'best_phone_model.pth')
        print(f"✨ 최고 AUC 갱신! 모델 저장 완료: {best_val_auc:.4f}")

    # 로그 출력 (Validation Loss 포함)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{EPOCHS}] | LR: {current_lr:.6f}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}")
    print("-" * 50)

✨ 최고 AUC 갱신! 모델 저장 완료: 0.9435
Epoch [1/30] | LR: 0.000002
Train Loss: 0.2611 | Val Loss: 0.2355 | Val AUC: 0.9435
--------------------------------------------------
Epoch [2/30] | LR: 0.000010
Train Loss: 0.2911 | Val Loss: 0.2338 | Val AUC: 0.9435
--------------------------------------------------
Epoch [3/30] | LR: 0.000021
Train Loss: 0.2791 | Val Loss: 0.2326 | Val AUC: 0.9435
--------------------------------------------------
✨ 최고 AUC 갱신! 모델 저장 완료: 0.9441
Epoch [4/30] | LR: 0.000035
Train Loss: 0.2718 | Val Loss: 0.2245 | Val AUC: 0.9441
--------------------------------------------------
✨ 최고 AUC 갱신! 모델 저장 완료: 0.9507
Epoch [5/30] | LR: 0.000050
Train Loss: 0.2672 | Val Loss: 0.2174 | Val AUC: 0.9507
--------------------------------------------------
Epoch [6/30] | LR: 0.000065
Train Loss: 0.2676 | Val Loss: 0.2333 | Val AUC: 0.9402
--------------------------------------------------
Epoch [7/30] | LR: 0.000079
Train Loss: 0.2729 | Val Loss: 0.2253 | Val AUC: 0.9420
----------------

In [9]:
# ============================================
# [Step 5] 추론 및 submission.csv 생성
# ============================================

from transformers import ViTForImageClassification
import torch.nn as nn # nn 모듈이 이전에 임포트되지 않았을 경우를 대비하여 추가

model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

# 2. 마지막 분류 레이어 교체 (출력 1개: Galaxy vs iPhone)
# 모델의 classifier 레이어를 1개의 출력으로 다시 설정합니다.
model.classifier = nn.Linear(model.classifier.in_features, 1)

# 3. 저장된 가중치('best_phone_model.pth') 입히기
model_path = 'best_phone_model.pth'
state_dict = torch.load(model_path, map_location=device)

# 만약 모델 전체가 아닌 state_dict만 저장했다면 아래 코드를 사용합니다.
model.load_state_dict(state_dict)

# 4. 모델을 GPU/CPU로 이동 및 평가 모드 전환
model.to(device)
model.eval()

print(f"✅ ViT 모델 가중치 로드 완료: {model_path}")

print("🔍 Test 데이터 추론 중...")

model.eval()
test_preds = []

with torch.no_grad():
    for imgs in test_loader:
        imgs = imgs.to(device)
        probs = torch.sigmoid(model(imgs).logits.squeeze()) # .logits 추가
        # 배치 사이즈가 1일 경우를 대비해 리스트로 변환 시 차원 일치
        if probs.dim() == 0:
            test_preds.append(probs.item())
        else:
            test_preds.extend(probs.cpu().numpy())

# 정해진 제출 양식(Image_ID, Target)에 맞춰 DataFrame 생성
submission = pd.DataFrame({
    'Image_ID': test_fnames,
    'Target': test_preds
})

# CSV 파일로 저장
submission.to_csv('/content/submission.csv', index=False)

print("✅ 최종 제출 파일(submission.csv) 생성 완료!")
print(submission.head())

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✅ ViT 모델 가중치 로드 완료: best_phone_model.pth
🔍 Test 데이터 추론 중...
✅ 최종 제출 파일(submission.csv) 생성 완료!
        Image_ID    Target
0  test_0001.jpg  0.947057
1  test_0002.jpg  0.800767
2  test_0003.jpg  0.899986
3  test_0004.jpg  0.958390
4  test_0005.jpg  0.857965
